# IMPORTACIONES

In [31]:
from google.colab import drive
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

# CONEXIÓN DE DRIVE

In [32]:
# Montamos Drive en nuestro espacio de trabajo
drive.mount('/content/drive')

# Cargamos los datos en un Dataframe
data = pd.read_csv('/content/drive/MyDrive/DIET-IA/dataset/13k-recipes.csv')

# Visualizamos las primeras filas para verificar alérgenos e ingredientes
print(data.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Unnamed: 0                                              Title  \
0           0  Miso-Butter Roast Chicken With Acorn Squash Pa...   
1           1                    Crispy Salt and Pepper Potatoes   
2           2                        Thanksgiving Mac and Cheese   
3           3                 Italian Sausage and Bread Stuffing   
4           4                                       Newton's Law   

                                         Ingredients  \
0  ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...   
1  ['2 large egg whites', '1 pound new potatoes (...   
2  ['1 cup evaporated milk', '1 cup whole milk', ...   
3  ['1 (¾- to 1-pound) round Italian loaf, cut in...   
4  ['1 teaspoon dark brown sugar', '1 teaspoon ho...   

                                        Instructions  \
0  Pat chicken dry with paper towels, season all ...   
1  Preheat ov

# Modelo KNN

In [33]:


# =========================
# 1. PREPARAR DATOS
# =========================
df = data.copy()
df = df.dropna(subset=['Title', 'Cleaned_Ingredients', 'Instructions'])

def join_ingredients(ing_str):
    try:
        lista = ast.literal_eval(ing_str)
        return " ".join(lista)
    except:
        return str(ing_str)

df['Features'] = df['Cleaned_Ingredients'].apply(join_ingredients)

# Filtrar recetas muy raras (estabilidad)
df = df[df['Title'].map(df['Title'].value_counts()) > 1]

# =========================
# 2. CREAR EMBEDDINGS
# =========================
print("Generando embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    df['Features'].tolist(),
    show_progress_bar=True
)

# =========================
# 3. ENTRENAR BUSCADOR VECINAL
# =========================
nn = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)

nn.fit(embeddings)

# =========================
# 4. FUNCIÓN FINAL TOP-5
# =========================
def diet_ia_recommender(ingredientes_usuario, top_k=5):
    emb = embedder.encode([ingredientes_usuario])
    distances, indices = nn.kneighbors(emb, n_neighbors=top_k)

    resultados = []

    for rank, idx in enumerate(indices[0]):
        receta = df.iloc[idx]

        resultados.append({
            "Rank": rank + 1,
            "Similitud": float(1 - distances[0][rank]),
            "Receta": receta['Title'],
            "Ingredientes": receta['Ingredients'],
            "Instrucciones": receta['Instructions']
        })

    return resultados

Generando embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [34]:
diet_ia_recommender("rice, chicken, potato, garlic")

[{'Rank': 1,
  'Similitud': 0.6387215852737427,
  'Receta': 'Chicken Tikka Masala',
  'Ingredientes': "['6 garlic cloves, finely grated', '4 teaspoons finely grated peeled ginger', '4 teaspoons ground turmeric', '2 teaspoons garam masala', '2 teaspoons ground coriander', '2 teaspoons ground cumin', '1 1/2 cups whole-milk yogurt (not Greek)', '1 tablespoon kosher salt', '2 pounds skinless, boneless chicken breasts, halved lengthwise', '3 tablespoons ghee (clarified butter) or vegetable oil', '1 small onion, thinly sliced', '1/4 cup tomato paste', '6 cardamom pods, crushed', '2 dried chiles de árbol or 1/2 teaspoon crushed red pepper flakes', '1 28-ounce can whole peeled tomatoes', '2 cups heavy cream', '3/4 cup chopped fresh cilantro plus sprigs for garnish', 'Steamed basmati rice (for serving)']",
  'Instrucciones': 'Combine garlic, ginger, turmeric, garam masala, coriander, and cumin in a small bowl. Whisk yogurt, salt, and half of spice mixture in a medium bowl; add chicken and turn 